# Part 4: Leak-Free CAMELS Feature Engineering & Forward Failure Labeling
Builds 26 point-in-time CAMELS, Texas Ratio, and 4-quarter historical lag features joined with the 4-quarter forward failure indicator $y_{i,q} = 1\{\text{fail in } (q, q+4]\}$, and verifies the causal leak-free truncation property.

In [1]:
import sys, datetime
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import polars as pl
from src.bfews.features import FEATURE_COLS_CAMELS, compute_camels_features

print(f'Total CAMELS features ({len(FEATURE_COLS_CAMELS)}):', FEATURE_COLS_CAMELS)
print(pl.read_csv('../outputs/tables/gold_split_summary.csv'))

Total CAMELS features (26): ['eq_asset_ratio', 'rbc1aaj_clean', 'rbcrwaj_clean', 'cap_distress_flag', 'nclnls_ratio', 'naasset_ratio', 'ore_ratio', 'lnatres_ratio', 'texas_ratio', 'efficiency_ratio', 'roa_ytd', 'nimy_clean', 'provision_asset_ratio', 'netinc_q_asset_ratio', 'liquid_asset_ratio', 'brokered_dep_ratio', 'loan_to_deposit_ratio', 're_loan_share', 'ci_loan_share', 'con_loan_share', 'log_asset', 'asset_growth_4q', 'loan_growth_4q', 'd4_texas_ratio', 'd4_nclnls_ratio', 'd4_eq_ratio']
shape: (8, 9)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ split_grou ┆ rows    ┆ banks ┆ positive_r ┆ … ┆ positive_r ┆ first_quar ┆ last_quart ┆ scope     │
│ p          ┆ ---     ┆ ---   ┆ ows        ┆   ┆ ate_pct    ┆ ter        ┆ er         ┆ ---       │
│ ---        ┆ i64     ┆ i64   ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ str       │
│ str        ┆         ┆       ┆ i64        ┆   ┆ f64        ┆ str        ┆ str      

In [2]:
# Live verification of leak-free causal truncation property
dates = [datetime.date(2020, 3, 31), datetime.date(2020, 6, 30), datetime.date(2020, 9, 30), datetime.date(2020, 12, 31),
         datetime.date(2021, 3, 31), datetime.date(2021, 6, 30), datetime.date(2021, 9, 30), datetime.date(2021, 12, 31)]
n = len(dates)
sample_df = pl.DataFrame({
    'CERT': [101]*n, 'REPDTE': dates, 'ASSET': [1000.0 + i*50 for i in range(n)],
    'EQ': [100.0 + i*5 for i in range(n)], 'LNLSGR': [600.0 + i*30 for i in range(n)],
    'LNLSNET': [580.0 + i*30 for i in range(n)], 'NCLNLS': [10.0 + i for i in range(n)],
    'ORE': [2.0]*n, 'LNATRES': [20.0]*n, 'DEP': [800.0]*n, 'CHBAL': [50.0]*n, 'SC': [100.0]*n,
    'LNRE': [300.0]*n, 'LNCI': [150.0]*n, 'LNCON': [150.0]*n, 'ROA': [1.0]*n, 'ROE': [10.0]*n,
    'NIMY': [3.5]*n, 'RBC1AAJ': [12.0]*n, 'RBCRWAJ': [14.0]*n, 'NONIX': [30.0]*n,
    'INTINC': [40.0]*n, 'EINTEXP': [10.0]*n, 'NONII': [5.0]*n, 'ELNATR_Q': [2.0]*n,
    'NETINC_Q': [10.0]*n, 'NAASSET': [5.0]*n, 'BRO': [10.0]*n
})
f_full = compute_camels_features(sample_df).filter(pl.col('REPDTE') == datetime.date(2020, 12, 31))
f_trunc = compute_camels_features(sample_df.slice(0, 4)).filter(pl.col('REPDTE') == datetime.date(2020, 12, 31))
diffs = {c: abs((f_full[c].item() or 0.0) - (f_trunc[c].item() or 0.0)) for c in ['eq_asset_ratio', 'texas_ratio', 'nclnls_ratio']}
print('Max abs diff at 2020-12-31 between full (2020-2021) and truncated (2020 only) panel:', diffs)
assert max(diffs.values()) == 0.0, 'Future leakage detected!'

17:36:11 features | Computing CAMELS ratios and Texas ratio...


17:36:11 features | Computing 4-quarter backward-looking dynamics (guarded: lag must be exactly 4 quarters)...


17:36:11 features | Computing CAMELS ratios and Texas ratio...


17:36:11 features | Computing 4-quarter backward-looking dynamics (guarded: lag must be exactly 4 quarters)...


Max abs diff at 2020-12-31 between full (2020-2021) and truncated (2020 only) panel: {'eq_asset_ratio': 0.0, 'texas_ratio': 0.0, 'nclnls_ratio': 0.0}
